# Hyperparameter Optimization and Model Ensemble

Following the feature engineering phase, this notebook focuses on fine-tuning high-performance Gradient Boosting Decision Trees (GBDTs) using Optuna.

We leverage the pre-processed hybrid feature sets (Lexical, Semantic, and Similarity) to maximize the model's predictive power for community rule violations.

## 1. Data Loading and Artifact Retrieval

In this section, we import the serialized artifacts generated in the previous stage. Loading the data in Compressed Sparse Row (CSR) format ensures optimal memory management, allowing the system to focus resources on the optimization trials.

In [1]:
import scipy.sparse as sp
import joblib
import os

# Define artifact directory
data_dir = "data/processed"

# Load serialized feature matrices and metadata
X_train_csr = sp.load_npz(os.path.join(data_dir, 'X_train_csr.npz'))
X_test_csr = sp.load_npz(os.path.join(data_dir, 'X_test_csr.npz'))
y_train = joblib.load(os.path.join(data_dir, 'y_train.pkl'))
test_ids = joblib.load(os.path.join(data_dir, 'test_ids.pkl'))

## 2. Hyperparameter Optimization (Optuna)

To ensure reliable validation, we use GroupKFold (grouped by subreddit), preventing data leakage between training and validation sets.

In [2]:
#!pip install optuna
import optuna
import gc
import numpy as np
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
#!pip install catboost
from catboost import CatBoostClassifier

from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)

import joblib
groups = joblib.load('data/processed/groups.pkl')

### LGBM Objective

In [20]:
def objective_lgbm(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "random_state": 42,
        "num_leaves": trial.suggest_int("num_leaves", 31, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "n_estimators": 1000,
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1, # Eklendi: subsample'ın aktif olması için gerekli
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    auc_scores = []
    for train_idx, val_idx in gkf.split(X_train_csr, y_train, groups=groups):
        X_tr, X_val = X_train_csr[train_idx], X_train_csr[val_idx]
        y_tr, y_val = y_train.values[train_idx], y_train.values[val_idx]

        model = LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                early_stopping(stopping_rounds=50),
                log_evaluation(period=0)
            ]
        )

        preds = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, preds))

        del X_tr, X_val, y_tr, y_val
        gc.collect()

    return np.mean(auc_scores)

### XGBoost Objective

In [21]:
def objective_xgb(trial):
    params = {
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "tree_method": "hist",
        "device": "cuda",
        "random_state": 42,
        "early_stopping_rounds": 50
    }

    auc_scores = []
    for train_idx, val_idx in gkf.split(X_train_csr, y_train, groups=groups):
        X_tr, X_val = X_train_csr[train_idx], X_train_csr[val_idx]
        y_tr, y_val = y_train.values[train_idx], y_train.values[val_idx]

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        preds = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, preds))

        del X_tr, X_val, y_tr, y_val
        gc.collect()

    return np.mean(auc_scores)

### CatBoost Objective

In [9]:
def objective_cat(trial):
    params = {
        "iterations": 1000,
        "depth": trial.suggest_int("depth", 3, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 5),
        "task_type": "GPU",
        "random_seed": 42,
        "verbose": False,
        "early_stopping_rounds": 50
    }

    auc_scores = []
    for train_idx, val_idx in gkf.split(X_train_csr, y_train, groups=groups):
        X_tr, X_val = X_train_csr[train_idx], X_train_csr[val_idx]
        y_tr, y_val = y_train.values[train_idx], y_train.values[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        preds = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, preds))

        del X_tr, X_val, y_tr, y_val
        gc.collect()

    return np.mean(auc_scores)

Run LGBM Study

In [23]:
study_lgbm = optuna.create_study(direction="maximize")

[I 2026-02-15 15:51:57,887] A new study created in memory with name: no-name-ecc630a2-1483-490f-9af3-462d0ec8a3dd


In [24]:
study_lgbm.optimize(objective_lgbm, n_trials=10)
print(f"LGBM Best Score: {study_lgbm.best_value}")
print(f"Best Parameters: {study_lgbm.best_trial.params}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[242]	valid_0's auc: 0.778499


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[76]	valid_0's auc: 0.724391


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[258]	valid_0's auc: 0.8799


...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[322]	valid_0's auc: 0.77119


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[95]	valid_0's auc: 0.80961


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-02-15 16:48:14,188] Trial 9 finished with value: 0.7822504380397468 and parameters: {'num_leaves': 160, 'max_depth': 6, 'learning_rate': 0.013943018926228734, 'subsample': 0.7429467571604556, 'colsample_bytree': 0.6047204709688686, 'reg_alpha': 8.817549219308187, 'reg_lambda': 0.0034647728107076673}. Best is trial 3 with value: 0.7969326726509879.


LGBM Best Score: 0.7969326726509879
Best Parameters: {'num_leaves': 90, 'max_depth': 12, 'learning_rate': 0.05788536024938373, 'subsample': 0.8659995717946736, 'colsample_bytree': 0.8704872244974341, 'reg_alpha': 0.0060150916672568645, 'reg_lambda': 0.003449486887507533}


Run XGBoost Study

In [25]:
study_xgb = optuna.create_study(direction="maximize")

[I 2026-02-15 16:50:57,354] A new study created in memory with name: no-name-40354239-e32a-4e1a-b6fa-71aa90a36fa4


In [26]:
study_xgb.optimize(objective_xgb, n_trials=10)
print(f"XGB Best Score: {study_xgb.best_value}")
print(f"Best Parameters: {study_xgb.best_trial.params}")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [16:51:15] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-02-15 16:52:19,472] Trial 0 finished with value: 0.7753192344197806 and parameters: {'max_depth': 9, 'learning_rate': 0.07575818336532504, 'subsample': 0.839687000786997, 'colsample_bytree': 0.5164584291652223, 'gamma': 0.9369808679202218, 'reg_alpha': 1.6005508392886738, 'reg_lambda': 8.378043550183738}. Best is trial 0 with value: 0.7753192344197806.
[I 2026-02-15 16:53:19,920] Trial 1 finished with value: 0.7762891032009847 and parameters: 

XGB Best Score: 0.798248269930351
Best Parameters: {'max_depth': 5, 'learning_rate': 0.018074966193584112, 'subsample': 0.8022176753909978, 'colsample_bytree': 0.8592655880039336, 'gamma': 4.940705104411184, 'reg_alpha': 0.004552355865136355, 'reg_lambda': 0.0057379022363263504}


Run CatBoost Study

In [10]:
study_cat = optuna.create_study(direction="maximize")

[I 2026-02-15 18:41:16,118] A new study created in memory with name: no-name-f1a3264f-156d-484f-9ff2-05981a1ce40b


In [11]:
study_cat.optimize(objective_cat, n_trials=3)
print(f"CatBoost Best Score: {study_cat.best_value}")
print(f"Best Parameters: {study_cat.best_trial.params}")

[I 2026-02-15 18:43:26,391] Trial 0 finished with value: 0.7902866649933389 and parameters: {'depth': 3, 'learning_rate': 0.07085996102844018, 'l2_leaf_reg': 6.606716273438306, 'bagging_temperature': 0.63253970775277}. Best is trial 0 with value: 0.7902866649933389.
[I 2026-02-15 18:47:36,949] Trial 1 finished with value: 0.7827002700511505 and parameters: {'depth': 5, 'learning_rate': 0.059463391213426815, 'l2_leaf_reg': 9.586884246072419, 'bagging_temperature': 1.3870160731796743}. Best is trial 0 with value: 0.7902866649933389.
[I 2026-02-15 18:51:43,848] Trial 2 finished with value: 0.799364663125847 and parameters: {'depth': 3, 'learning_rate': 0.020930638028573506, 'l2_leaf_reg': 3.399105945052233, 'bagging_temperature': 0.784044217483128}. Best is trial 2 with value: 0.799364663125847.


CatBoost Best Score: 0.799364663125847
Best Parameters: {'depth': 3, 'learning_rate': 0.020930638028573506, 'l2_leaf_reg': 3.399105945052233, 'bagging_temperature': 0.784044217483128}


## 3. Exporting Best Hyperparameters
We consolidate the best parameters from all Optuna studies and export them to a JSON file. This ensures modularity, allowing the final training notebook to load these parameters instantly without re-running the optimization trials.

This code block can also be used for saved parameters:

```bash
best_params_all = {
    "lgbm": study_lgbm.best_trial.params,
    "xgb": study_xgb.best_trial.params,
    "cat": study_cat.best_trial.params
}

In [13]:
import json
import os

best_params_all = {
    "lgbm": {
        'num_leaves': 90,
        'max_depth': 12,
        'learning_rate': 0.05788536024938373,
        'subsample': 0.8659995717946736,
        'colsample_bytree': 0.8704872244974341,
        'reg_alpha': 0.0060150916672568645,
        'reg_lambda': 0.003449486887507533
    },
    "xgb": {
        'max_depth': 5,
        'learning_rate': 0.018074966193584112,
        'subsample': 0.8022176753909978,
        'colsample_bytree': 0.8592655880039336,
        'gamma': 4.940705104411184,
        'reg_alpha': 0.004552355865136355,
        'reg_lambda': 0.0057379022363263504
    },
    "cat": {
        'depth': 3,
        'learning_rate': 0.020930638028573506,
        'l2_leaf_reg': 3.399105945052233,
        'bagging_temperature': 0.784044217483128
    }
}

output_path = 'data/processed/best_hyperparameters.json'
os.makedirs('data/processed', exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(best_params_all, f, indent=4)

## 4. Local Download (Optional)

This step allows you to download the optimized hyperparameters to your local machine for persistence and use in subsequent notebooks.

```bash
# from google.colab import files
# files.download('data/processed/best_hyperparameters.json')